# DINOv3 Distillation to YOLO11l-seg

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/marjanstoimcev/DINOv3distill/blob/main/dinov3_distillation.ipynb)

This notebook demonstrates how to distill knowledge from DINOv3 (ViT-S/16) to YOLO11l-seg using LightlyTrain.

**Workflow:**
1. Install dependencies
2. Download TACO dataset
3. **Pretrain** YOLO11l-seg backbone using DINOv3 distillation (unlabeled images)
4. **Fine-tune** on labeled instance segmentation task (polygon masks)

**Compare with:** `train_from_scratch.ipynb` (baseline without pretraining)

**Model:**
- YOLO11l-seg: 27.7M params, 143.0 GFLOPs

**Dataset:** TACO - Trash Annotations in Context
- 59 classes of litter/trash
- 1,499 images with polygon segmentation masks
- License: CC BY 4.0

## 1. Install Dependencies

In [ ]:
# Install lightly-train with ultralytics support
!pip install -q "lightly-train[ultralytics]" gdown

# Verify installation
import lightly_train
print(f"LightlyTrain version: {lightly_train.__version__}")

In [ ]:
# Additional imports
import os
import shutil
from pathlib import Path
from ultralytics import YOLO

# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Hyperparameters

Configure training hyperparameters here. Defaults based on [LightlyTrain recommendations](https://docs.lightly.ai/train/).

In [ ]:
# ============================================================
# HYPERPARAMETERS - Modify these as needed
# ============================================================

# Distillation pretraining
DISTILL_EPOCHS = 100        # LightlyTrain recommends 100-300 for DINOv2/v3
DISTILL_BATCH_SIZE = 128    # Recommended: 128-1536 (adjust for GPU memory)
DISTILL_LR = None           # None = auto-scaled based on batch size

# Fine-tuning
FINETUNE_EPOCHS = 50        # Ultralytics default for fine-tuning
FINETUNE_BATCH_SIZE = 16    # Adjust based on GPU memory
FINETUNE_LR = 0.01          # Ultralytics default
FINETUNE_LRF = 0.01         # Final LR = LR * LRF

# Common
IMAGE_SIZE = 640            # Input image size
PATIENCE = 10               # Early stopping patience

# Model
STUDENT_MODEL = "ultralytics/yolo11l-seg.yaml"  # 27.7M params
TEACHER_MODEL = "dinov3/vits16"                  # DINOv3 ViT-S/16

print("Hyperparameters:")
print(f"  Distillation: {DISTILL_EPOCHS} epochs, batch={DISTILL_BATCH_SIZE}, lr={DISTILL_LR or 'auto'}")
print(f"  Fine-tuning:  {FINETUNE_EPOCHS} epochs, batch={FINETUNE_BATCH_SIZE}, lr={FINETUNE_LR}")
print(f"  Image size:   {IMAGE_SIZE}")
print(f"  Student:      {STUDENT_MODEL}")
print(f"  Teacher:      {TEACHER_MODEL}")

## 3. Dataset Setup

**TACO: Trash Annotations in Context** - Instance segmentation dataset of litter in the wild.

59 classes including:
- Plastic bottles, cups, bags, wrappers
- Glass bottles, jars
- Metal cans, foil, bottle caps
- Paper, cardboard, cartons
- Cigarettes, food waste, and more

Dataset splits:
- Train: 1,049 images
- Valid: 299 images  
- Test: 151 images

Source: [Roboflow Universe](https://universe.roboflow.com/mohamed-traore-2ekkp/taco-trash-annotations-in-context)

In [ ]:
# Download TACO dataset from Google Drive
import gdown
import zipfile

# Google Drive file ID from the shared link
# Link: https://drive.google.com/file/d/1HZBR53rs_igtr2ZlYqsLrS0uROCcTqzl/view?usp=sharing
FILE_ID = "1HZBR53rs_igtr2ZlYqsLrS0uROCcTqzl"
ZIP_PATH = "taco.zip"
DATA_PATH = Path("./data/taco")

if not DATA_PATH.exists():
    print("Downloading TACO dataset from Google Drive...")
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('./data')
    print("Dataset ready!")
else:
    print(f"Dataset already exists at {DATA_PATH}")

!ls -la ./data/taco/

In [ ]:
# Set the dataset path
dataset_path = DATA_PATH
print(f"Using dataset: {dataset_path}")

In [ ]:
# Verify dataset structure
print("Dataset structure:")
print(f"  Root: {dataset_path}")

total_images = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    labels_path = dataset_path / split / 'labels'
    
    if images_path.exists():
        num_images = len(list(images_path.glob('*.jpg'))) + len(list(images_path.glob('*.png')))
        total_images += num_images
        print(f"  {split}/images: {num_images} images")
    
    if labels_path.exists():
        num_labels = len(list(labels_path.glob('*.txt')))
        print(f"  {split}/labels: {num_labels} labels")

print(f"\nTotal images: {total_images}")

In [ ]:
# Create a directory with all images for distillation (unlabeled pretraining)
# Distillation only needs images, no labels!

UNLABELED_DIR = Path("./unlabeled_images")
UNLABELED_DIR.mkdir(exist_ok=True)

# Copy all images from train, valid, test to unlabeled directory
# For distillation, we use ALL available images (labels are ignored)

total_copied = 0
for split in ['train', 'valid', 'test']:
    images_path = dataset_path / split / 'images'
    if images_path.exists():
        for img_file in images_path.glob('*'):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                dest = UNLABELED_DIR / img_file.name
                if not dest.exists():
                    shutil.copy2(img_file, dest)
                    total_copied += 1

print(f"Copied {total_copied} images")
print(f"Total images for distillation: {len(list(UNLABELED_DIR.glob('*')))}")

## 4. DINOv3 Distillation to YOLO11l-seg

Now we pretrain the YOLO11l-seg backbone using knowledge distillation from DINOv3.

**Key parameters:**
- `model`: YOLO11l-seg (student model, 27.7M params)
- `teacher`: DINOv3 ViT-S/16 (teacher model)
- `method`: distillation
- `data`: Path to unlabeled images

In [ ]:
# Configure distillation parameters
import lightly_train

# Output directory for pretrained model
OUTPUT_DIR = "./output/dinov3_yolo11l_distill"

# Build config
distill_config = {
    "out": OUTPUT_DIR,
    "data": str(UNLABELED_DIR),
    "model": STUDENT_MODEL,
    "method": "distillation",
    "method_args": {
        "teacher": TEACHER_MODEL,
    },
    "epochs": DISTILL_EPOCHS,
    "batch_size": DISTILL_BATCH_SIZE,
}

# Add learning rate if specified
if DISTILL_LR is not None:
    distill_config["lr"] = DISTILL_LR

print("Distillation Configuration:")
for k, v in distill_config.items():
    print(f"  {k}: {v}")

In [ ]:
# Run distillation pretraining
# This will take a while depending on your GPU and number of epochs

print("Starting DINOv3 -> YOLO11l-seg distillation...")
print(f"Using {len(list(UNLABELED_DIR.glob('*')))} unlabeled images")
print("="*60)

lightly_train.pretrain(**distill_config)

print("="*60)
print("Distillation complete!")

In [ ]:
# Check the output
output_path = Path(OUTPUT_DIR)

print("Output files:")
for f in output_path.rglob('*'):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.relative_to(output_path)}: {size_mb:.1f} MB")

## 5. Fine-tune for Instance Segmentation

Now we load the pretrained backbone and fine-tune on the labeled TACO dataset for instance segmentation with polygon masks.

In [ ]:
# Path to the pretrained model
PRETRAINED_MODEL_PATH = Path(OUTPUT_DIR) / "exported_models" / "exported_last.pt"

# Alternative: if the path structure is different
if not PRETRAINED_MODEL_PATH.exists():
    # Try to find the exported model
    possible_paths = list(Path(OUTPUT_DIR).rglob("*.pt"))
    print("Available .pt files:")
    for p in possible_paths:
        print(f"  {p}")
    if possible_paths:
        PRETRAINED_MODEL_PATH = possible_paths[0]

print(f"Using pretrained model: {PRETRAINED_MODEL_PATH}")

In [ ]:
# Load the pretrained model with Ultralytics
from ultralytics import YOLO

# Load pretrained backbone
model = YOLO(str(PRETRAINED_MODEL_PATH))

print(f"Model loaded: {model.model.__class__.__name__}")
print(f"Task: {model.task}")

In [ ]:
# Use existing data.yaml from TACO dataset
DATA_YAML = dataset_path / "data.yaml"

if DATA_YAML.exists():
    print(f"Using existing data.yaml: {DATA_YAML}")
    with open(DATA_YAML, 'r') as f:
        content = f.read()
        # Show first 30 lines (class names)
        lines = content.split('\n')
        for line in lines[:35]:
            print(line)
        if len(lines) > 35:
            print("...")
else:
    print("ERROR: data.yaml not found!")
    print("Please ensure the TACO dataset is properly downloaded.")

In [ ]:
# Fine-tune for instance segmentation
# The pretrained backbone should give better results than training from scratch

FINETUNE_OUTPUT = "./output/finetune_seg"

results = model.train(
    data=str(DATA_YAML),
    epochs=FINETUNE_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=FINETUNE_BATCH_SIZE,
    lr0=FINETUNE_LR,
    lrf=FINETUNE_LRF,
    project=FINETUNE_OUTPUT,
    name="yolo11l_seg_taco",
    patience=PATIENCE,
    save=True,
    plots=True,
)

## 6. Evaluate Model

In [ ]:
# Validate the fine-tuned model
metrics = model.val()

print("\n" + "="*60)
print("RESULTS: DINOv3 Distillation + Fine-tuning")
print("="*60)
print(f"\nBox Detection:")
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"\nInstance Segmentation:")
print(f"  mAP50:    {metrics.seg.map50:.4f}")
print(f"  mAP50-95: {metrics.seg.map:.4f}")
print("="*60)

In [ ]:
# Save metrics for comparison
import json

metrics_dict = {
    "model": "YOLO11l-seg",
    "training": "dinov3_distillation",
    "hyperparameters": {
        "distill_epochs": DISTILL_EPOCHS,
        "distill_batch_size": DISTILL_BATCH_SIZE,
        "distill_lr": DISTILL_LR,
        "finetune_epochs": FINETUNE_EPOCHS,
        "finetune_batch_size": FINETUNE_BATCH_SIZE,
        "finetune_lr": FINETUNE_LR,
        "image_size": IMAGE_SIZE,
    },
    "box_map50": float(metrics.box.map50),
    "box_map": float(metrics.box.map),
    "seg_map50": float(metrics.seg.map50),
    "seg_map": float(metrics.seg.map),
}

# Ensure output directory exists
Path(FINETUNE_OUTPUT).mkdir(parents=True, exist_ok=True)

with open(f"{FINETUNE_OUTPUT}/metrics_distillation.json", "w") as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Metrics saved to: {FINETUNE_OUTPUT}/metrics_distillation.json")
print(json.dumps(metrics_dict, indent=2))

## 7. Inference Demo

In [ ]:
# Run inference on test images
import matplotlib.pyplot as plt
from PIL import Image
import random

# Get some test images
test_images_path = dataset_path / "test" / "images"
test_images = list(test_images_path.glob("*.jpg"))[:5]

if not test_images:
    test_images = list(test_images_path.glob("*.png"))[:5]

print(f"Running inference on {len(test_images)} test images...")

In [ ]:
# Visualize predictions with segmentation masks
fig, axes = plt.subplots(1, min(5, len(test_images)), figsize=(20, 4))

if len(test_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, test_images):
    results = model.predict(str(img_path), verbose=False)
    result_img = results[0].plot()
    
    ax.imshow(result_img[:, :, ::-1])
    ax.axis('off')
    ax.set_title(img_path.name[:20] + "...")

plt.suptitle("YOLO11l-seg (DINOv3 Distillation) - Predictions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{FINETUNE_OUTPUT}/inference_distillation.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Export Model

In [ ]:
# Export to ONNX for deployment
model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True)
print("Model exported to ONNX format")

In [ ]:
# Download the trained model (for Colab)
from google.colab import files

# Find the best model
best_model_path = Path(FINETUNE_OUTPUT) / "yolo11l_seg_taco" / "weights" / "best.pt"

if best_model_path.exists():
    print(f"Downloading {best_model_path}...")
    files.download(str(best_model_path))
else:
    print("Best model not found. Available files:")
    for f in Path(FINETUNE_OUTPUT).rglob("*.pt"):
        print(f"  {f}")

## Summary

This notebook trained **YOLO11l-seg with DINOv3 distillation pretraining** on the TACO dataset.

**Two-stage training:**
1. **Distillation**: Transfer knowledge from DINOv3 → YOLO11l-seg backbone (unlabeled)
2. **Fine-tuning**: Train on labeled instance segmentation data

**Compare with:** `train_from_scratch.ipynb` to see the benefit of distillation pretraining.

**Expected benefit:** Better mAP scores, faster convergence, better generalization on small datasets.

**Dataset:** TACO (Trash Annotations in Context)
- Paper: [arXiv:2003.06975](https://arxiv.org/abs/2003.06975)
- License: CC BY 4.0